# H7 — OpenQuantum evidence and guarded hardware experiments
This notebook imports the production H0–H7 implementations. It compares exact classical enumeration, ideal Qiskit execution and a declared synthetic noise model. Hardware evidence is loaded only from a verified provider job receipt.

**Run All is offline by default.** Live connection, preparation and submission have separate switches. No paid credits are authorized. Classical validation remains authoritative; a cloud job is not a live landing controller or evidence of quantum advantage.

In [1]:
from pathlib import Path
import sys, json, time
from datetime import datetime, timezone, timedelta
import numpy as np
from qiskit import transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared/examples/recovery/hybrid_campaign.json").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from backend.app.hybrid_recovery import HybridRecoveryRequest, checkpoint_from_decision, prepare_problem
from backend.app.self_healing import RecoveryScenario, replay_recovery
from backend.app.hybrid_planning import solve_exact, validate_plan
from backend.app.hybrid_quantum import compile_qubo, compact_model, qaoa_statevector, run_qaoa
from quantum.openquantum_bridge import (OpenQuantumGateway, PreparedRun, build_qaoa_circuit, analyze_counts, choose_free_quote, resolve_hardware_recovery)
from shared.hybrid import fingerprint
LEDGER = Path.home() / ".local/share/lunar-h7-ledger"
print("Runtime ready; no cloud call made.")

Runtime ready; no cloud call made.


## 1. Replay the fault and rebuild the campaign
The synthetic H5 example produces a current checkpoint. H6 rechecks fuel, trajectories, exclusions and downstream slot timing. Site labels do not imply a real lunar terrain conversion.

In [2]:
raw = json.loads((ROOT / "shared/examples/recovery/hybrid_campaign.json").read_text())
raw.pop("quantum_budget")
physical_replay = replay_recovery(RecoveryScenario.model_validate(raw["recovery"]))
raw["checkpoint"] = checkpoint_from_decision(physical_replay["decisions"][-1]).model_dump(mode="json")
request = HybridRecoveryRequest.model_validate(raw)
prepared = prepare_problem(request)
problem = prepared["scenario"]
assert problem is not None
model = compact_model(compile_qubo(problem), max_qubits=2)
print({"logical_qubits": model.logical_qubits, "basis_states": len(model.one_hot_masks), "problem_fingerprint": fingerprint(problem), "landing_options": prepared["option_to_target"]})

{'logical_qubits': 2, 'basis_states': 4, 'problem_fingerprint': '176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3', 'landing_options': {'land-primary': 'primary', 'land-backup': 'backup', 'land-remote': 'remote'}}


## 2. Exact classical reference and bounded local training
Enumeration solves this four-state case exactly. Parameter training is classical and noiseless: two QAOA layers, 160 objective evaluations, fixed seeds. Its time is part of the hybrid method's cost.

In [3]:
start = time.perf_counter()
exact = solve_exact(problem, max_evaluations=4096)
classical_elapsed_s = time.perf_counter() - start
assert exact.status == "optimal"
trained = run_qaoa(problem, depth=2, evaluations=160, shots=1024, seed=17, max_qubits=2)
circuit = build_qaoa_circuit(problem, trained.gammas, trained.betas)
print({"exact_options": exact.plan.selected_option_ids, "exact_score": validate_plan(exact.plan, problem).exact_score, "classical_s": classical_elapsed_s, "training_and_sampling_s": trained.elapsed_s, "evaluations": trained.objective_evaluations, "logical_depth": circuit.depth(), "logical_gates": dict(circuit.count_ops()), "gammas": trained.gammas, "betas": trained.betas})

{'exact_options': ('follow-late', 'land-remote'), 'exact_score': '109982263595723410617/1000000000000000000', 'classical_s': 0.0007533750031143427, 'training_and_sampling_s': 0.034122375000151806, 'evaluations': 160, 'logical_depth': 12, 'logical_gates': {'rz': 6, 'cx': 4, 'rx': 4, 'h': 2, 'measure': 2}, 'gammas': (-1.5341939563645939, 1.4926101393076407), 'betas': (-0.6288668840612575, 0.7443484637598385)}


## 3. Independent circuit check
Qiskit's unitary statevector must agree with the existing NumPy H2 simulator. Qiskit bitstrings are little endian: the rightmost character is qubit zero. Raw samples are never repaired.

In [4]:
qiskit_state = Statevector.from_instruction(build_qaoa_circuit(problem, trained.gammas, trained.betas, measure=False)).data
reference = qaoa_statevector(model.normalized_energies, trained.gammas, trained.betas)
np.testing.assert_allclose(qiskit_state, reference, atol=1e-11)
print({"max_state_error": float(np.max(np.abs(qiskit_state-reference))), "ideal_optimal_probability": trained.optimal_probability, "uniform_optimal_probability": trained.uniform_optimal_probability})

{'max_state_error': 2.2729511820139823e-16, 'ideal_optimal_probability': 0.9641381165643415, 'uniform_optimal_probability': 0.25}


## 4. Ideal and synthetic-noise sampling
Assumptions: 0.1% single-qubit depolarization, 1% two-qubit depolarization, symmetric 2% readout error. These are illustrative values, **not device calibration**. The same seed-17 trained circuit is evaluated with three sampling seeds and 1024 shots each. Wilson intervals measure shot uncertainty only, not model error or device drift.

In [5]:
noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ["rz", "sx", "x"])
noise.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ["cx"])
noise.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.02, 0.98]]))
local_reports = []
for mode, noise_model in [("noiseless_simulator", None), ("noise_model", noise)]:
    simulator = AerSimulator(noise_model=noise_model)
    native = transpile(circuit, basis_gates=["rz", "sx", "x", "cx"], optimization_level=1, seed_transpiler=17)
    for seed in (17, 23, 41):
        start = time.perf_counter()
        counts = simulator.run(native, shots=1024, seed_simulator=seed).result().get_counts()
        report = analyze_counts(problem, counts, 1024, mode=mode)
        report.update(seed=seed, counts=counts, elapsed_s=time.perf_counter()-start)
        local_reports.append(report)
        print({k: report[k] for k in ("execution_mode", "seed", "counts", "sampled_feasible_rate", "sampled_optimal_rate", "optimal_rate_interval_95", "optimum_gap", "elapsed_s")})

{'execution_mode': 'noiseless_simulator', 'seed': 17, 'counts': {'01': 7, '00': 22, '10': 5, '11': 990}, 'sampled_feasible_rate': 0.9951171875, 'sampled_optimal_rate': 0.966796875, 'optimal_rate_interval_95': [0.9539609698563387, 0.9761435630906761], 'optimum_gap': 0.0, 'elapsed_s': 0.024697666995052714}
{'execution_mode': 'noiseless_simulator', 'seed': 23, 'counts': {'01': 2, '10': 4, '00': 21, '11': 997}, 'sampled_feasible_rate': 0.99609375, 'sampled_optimal_rate': 0.9736328125, 'optimal_rate_interval_95': [0.9619087646435117, 0.9818165459826435], 'optimum_gap': 0.0, 'elapsed_s': 0.019434667003224604}
{'execution_mode': 'noiseless_simulator', 'seed': 41, 'counts': {'01': 7, '00': 20, '10': 6, '11': 991}, 'sampled_feasible_rate': 0.994140625, 'sampled_optimal_rate': 0.9677734375, 'optimal_rate_interval_95': [0.9550881500366747, 0.9769622082930746], 'optimum_gap': 0.0, 'elapsed_s': 0.019369499997992534}
{'execution_mode': 'noise_model', 'seed': 17, 'counts': {'01': 30, '10': 36, '00': 

## 5. Parameter-seed sensitivity
This is a bounded sensitivity check, not a selection procedure based on hardware outcomes. The hardware circuit is fixed to seed 17 before submission.

In [6]:
for seed in (17, 23, 41):
    fit = run_qaoa(problem, depth=2, evaluations=160, shots=1024, seed=seed, max_qubits=2)
    print({"training_seed": seed, "ideal_optimal_probability": fit.optimal_probability, "sampled_optimal_rate": fit.sampled_optimal_rate, "elapsed_s": fit.elapsed_s})

{'training_seed': 17, 'ideal_optimal_probability': 0.9641381165643415, 'sampled_optimal_rate': 0.97265625, 'elapsed_s': 0.034311083996726666}
{'training_seed': 23, 'ideal_optimal_probability': 0.8218871010522649, 'sampled_optimal_rate': 0.822265625, 'elapsed_s': 0.035185332999390084}
{'training_seed': 41, 'ideal_optimal_probability': 0.924252124077921, 'sampled_optimal_rate': 0.9248046875, 'elapsed_s': 0.03508758300449699}


## 6. Optional read-only account connection
Set `CONNECT = True` only to query the account. On this machine, credentials are entered through the separate local `H7_connect_local.ipynb` with hidden fields and saved with user-only permissions outside Git. Other machines may use the official SDK environment variables. Never paste secrets into notebook source or outputs.

In [7]:
CONNECT = False
service = gateway = account = None
if CONNECT:
    from openquantum_sdk.auth import ClientCredentials
    from openquantum_sdk_qiskit import OpenQuantumService
    credential_path = Path.home() / ".config/lunar-openquantum/credentials.json"
    if credential_path.exists():
        service = OpenQuantumService(creds=ClientCredentials(**json.loads(credential_path.read_text())), auto_load=False)
    else:
        service = OpenQuantumService.from_env()
    gateway = OpenQuantumGateway(service, LEDGER)
    account = gateway.account_snapshot()
    print({k: account[k] for k in ("spark_credits", "full_credits", "checked_utc")})
    print([{k: b[k] for k in ("short_code", "type", "status", "accepting_jobs")} for b in account["backends"]])
else:
    print("Live account lookup disabled. No credit spent.")

Live account lookup disabled. No credit spent.


## 7. Quote only, then explicitly submit one bounded trial
A preparation uploads/transpiles the circuit and retrieves a quote; it does not create a running job. Use an online, accepting **QPU**. `ibm:qiskit` is a simulator.

Only Public / Standard is allowed. Spark must cover the quote, Full must be zero and Auto Refill must be visibly off. The SDK has no atomic Spark-only billing parameter, so this conservative gate blocks paid balances. Use one controlling caller; do not run parallel account spenders. The default trial permits **one job at a maximum of 2 credits**; new trial labels are new experiments and require a reviewed budget. No automatic job retry is performed.

In [8]:
PREPARE = False
BACKEND_CODE = "rigetti:cepheus-1-108q"
SHOTS = 256
TRIAL = "h7-smoke-256-v1"
MAX_CREDITS = 2
AUTO_REFILL_VERIFIED_OFF = False
if PREPARE:
    assert gateway is not None, "Enable and run the connection cell first"
    ticket = gateway.prepare(problem, circuit, backend_code=BACKEND_CODE, organization_id=account["organization_id"], shots=SHOTS, trial=TRIAL)
    preparation = service.scheduler.get_preparation_result(ticket.preparation_id)
    balance = service.management.get_credit_balance(ticket.organization_id)
    choice = choose_free_quote(preparation, balance, max_credits=MAX_CREDITS, auto_refill_disabled=AUTO_REFILL_VERIFIED_OFF)
    print({"backend": BACKEND_CODE, "shots": SHOTS, "quoted_spark_credits": choice["credits"], "submitted": False})
else:
    print("Preparation disabled.")

Preparation disabled.


In [9]:
SUBMIT = False
if SUBMIT:
    job_id = gateway.submit_free(ticket, max_credits=MAX_CREDITS, auto_refill_disabled=AUTO_REFILL_VERIFIED_OFF)
    print({"job_id": job_id, "status": "submitted; collect separately"})
else:
    print("Submission disabled; Run All cannot create a provider job.")

Submission disabled; Run All cannot create a provider job.


## 8. Collect without resubmitting
Rerun only the collection cell to check a queued job. If submission timed out, reconcile the account and saved receipt; **do not delete the ledger or generate a new trial label to retry**. After restarting the kernel, restore the saved `PreparedRun` manifest (parse `created_utc` with `datetime.fromisoformat`) and the job ID from its submission receipt, then collect.

In [10]:
COLLECT = False
if COLLECT:
    hardware_report = gateway.collect(ticket, job_id, problem)
    print(hardware_report)
else:
    print("Live collection disabled.")

Live collection disabled.


## 9. Recorded provider evidence and current fallback
Evidence is useful only with its matching input fingerprint and raw counts. Provider-backed QPU execution is not an independent cryptographic hardware attestation. The local report file contains no secrets or signed URLs. A late result may remain scientifically useful but must not replace a current landing decision.

In [11]:
recorded = []
for path in sorted(LEDGER.glob("result-*.json")) if LEDGER.exists() else []:
    item = json.loads(path.read_text())
    if item.get("problem_fingerprint") == fingerprint(problem):
        recorded.append(item)
if not recorded:
    print("No matching provider result recorded. Hardware benchmark pending.")
for item in recorded:
    print(item)
    if item.get("status") == "Completed" and item.get("execution_mode") == "hardware":
        now = datetime.now(timezone.utc)
        timely = resolve_hardware_recovery(request, item, deadline_utc=now + timedelta(seconds=30), now_utc=now)
        late = resolve_hardware_recovery(request, item, deadline_utc=now - timedelta(seconds=1), now_utc=now)
        print({"offline_acceptance": timely, "expired_acceptance": late})

{'job_id': '4c0adc79-6bd3-4578-9147-bbf8ab7b2209', 'status': 'Completed', 'backend_code': 'rigetti:cepheus-1-108q', 'problem_fingerprint': '176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3', 'circuit_sha256': '54275a0b2994acac9eb46f152aa9215d9ce12be4c922e24c4022dbb49c72298d', 'shots': 256, 'quoted_credits': 1.0, 'submitted_utc': '2026-11-14T20:10:08.326044+00:00', 'observed_utc': '2026-11-14T20:12:07.120647+00:00', 'execution_mode': 'hardware', 'counts': {'10': 9, '11': 200, '00': 18, '01': 29}, 'analysis': {'execution_mode': 'hardware', 'problem_fingerprint': '176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3', 'shots': 256, 'logical_qubits': 2, 'samples': [{'basis_index': 0, 'selected_option_ids': ['follow-early', 'land-backup'], 'count': 18, 'observed_frequency': 0.0703125, 'frequency_interval_95': [0.04493554746197091, 0.10839432915401236], 'feasible': True, 'optimal': False, 'exact_score': '49985116161374454691/1000000000000000000'}, {'basis_index': 

{'offline_acceptance': {'source': 'hardware', 'reason': 'hardware_accepted', 'selected_target_id': 'remote', 'plan': {'schema_version': 1, 'plan_id': 'h7-5639729c21b3f8f22a32f861a4d7b19cff081b715322728acb901a0ad00d8e50', 'scenario_id': 'h6-campaign', 'scenario_revision': 0, 'input_fingerprint': '176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3', 'state_revisions': {}, 'selected_option_ids': ['follow-late', 'land-remote'], 'execution_mode': 'hardware', 'solver_name': 'h7-observed-qaoa', 'validation_status': 'unvalidated'}, 'validation': {'status': 'feasible', 'input_fingerprint': '176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3', 'selected_option_ids': ['follow-late', 'land-remote'], 'issues': [], 'score': 109.98226359572341, 'exact_score': '109982263595723410617/1000000000000000000', 'metric_totals': {'recovery_propellant_kg': 17.736404276589383, 'utility': 110.0}}}, 'expired_acceptance': {'source': 'classical_fallback', 'reason': 'deadline', 'selected_

## 10. Offline flight replay with recorded hardware evidence
The provider result is replayed only at a matching current checkpoint. Initial or stale checkpoints use the classical fallback. Cloud execution has already finished; this demonstration pauses simulated time and does not claim an actual spacecraft waited for a cloud response.

In [12]:
completed = [r for r in recorded if r.get("status") == "Completed" and r.get("execution_mode") == "hardware"]
if completed:
    evidence = completed[-1]
    decisions = []
    def select_with_recorded_evidence(decision):
        current = request.model_copy(update={"checkpoint": checkpoint_from_decision(decision)})
        now = datetime.now(timezone.utc)
        resolved = resolve_hardware_recovery(current, evidence, deadline_utc=now + timedelta(seconds=30), now_utc=now)
        decisions.append(resolved)
        return resolved["selected_target_id"]
    replay = replay_recovery(request.recovery, target_selector=select_with_recorded_evidence)
    assert replay["status"] == "landed"
    assert decisions[-1]["source"] == "hardware"
    assert set(decisions[-1]["plan"]["selected_option_ids"]) == {"land-remote", "follow-late"}
    assert abs(replay["propellant_burned_kg"] + replay["propellant_lost_kg"] + replay["final_state"]["propellant_kg"] - 100) < 1e-8
    for index, decision in enumerate(decisions):
        print({"decision": index, "source": decision["source"], "reason": decision["reason"], "target": decision["selected_target_id"], "options": decision["plan"]["selected_option_ids"]})
    print({"status": replay["status"], "target": replay["selected_target_id"], "touchdown_utc": replay["final_state"]["epoch_utc"], "fuel_remaining_kg": replay["final_state"]["propellant_kg"], "simulated_clock_paused": True, "additional_provider_jobs": 0})
else:
    print("No completed hardware evidence: no hardware flight-replay claim.")

{'decision': 0, 'source': 'classical_fallback', 'reason': 'stale_snapshot', 'target': 'primary', 'options': ['follow-early', 'land-primary']}
{'decision': 1, 'source': 'hardware', 'reason': 'hardware_accepted', 'target': 'remote', 'options': ['follow-late', 'land-remote']}
{'status': 'landed', 'target': 'remote', 'touchdown_utc': '2030-01-01T00:00:34.999982Z', 'fuel_remaining_kg': 76.81443885247847, 'simulated_clock_paused': True, 'additional_provider_jobs': 0}


## 11. Interpretation
Compare feasibility, optimum gap, raw optimal-shot frequency and its interval, native gates, queue/total latency and actual credit debit. A higher success rate than uniform sampling is not quantum advantage: exact classical enumeration already solves this tiny example. Noise assumptions are illustrative, parameter training and physical screening are classical, and offline flight time is paused during cloud evaluation. Broader scaling, calibrated device noise, repeated hardware trials and live flight control remain separate work.

See `docs/openquantum-hardware.md` for recorded verification and sources.

Provider infrastructure: Open Quantum Public Plan; attribution follows [Open Quantum citation guidance](https://www.openquantum.com/citation). Kuhn, Armbruster and Wold (2026), [doi:10.5281/zenodo.22103261](https://doi.org/10.5281/zenodo.22103261). No provider endorsement is implied.